# Classification supervisée avec des Support Vector Machines (SVM)

Étude comparative de classifieurs **SVM** appliqués à plusieurs jeux de données de l'[UCI Machine Learning Repository](https://archive.ics.uci.edu/), classés du plus simple au plus complexe : *Breast Cancer*, *Spambase*, *Car Evaluation* et *Mushroom*.

L'objectif est d'observer concrètement comment le **choix du noyau** (linéaire, RBF, polynomial), la **normalisation des données**, la **sélection de variables** et l'**optimisation des hyperparamètres** influencent les performances d'un modèle SVM.

## Démarche

Un SVM cherche l'hyperplan qui sépare au mieux deux classes en **maximisant la marge** entre elles. C'est un modèle puissant mais **sensible à l'échelle des variables** (il repose sur des distances) et au **choix du noyau**, qui détermine sa capacité à séparer des données non linéairement séparables.

Pour bien isoler ces effets, on progresse volontairement du jeu de données le plus simple au plus difficile :

1. **Breast Cancer** — variables numériques, classification binaire : le cas le plus simple pour poser les bases.
2. **Spambase** — beaucoup de variables : on met l'accent sur la sélection de features.
3. **Car Evaluation** — variables catégorielles et cible multi-classes : il faut adapter l'encodage.
4. **Mushroom** — entièrement catégoriel et volumineux : on teste le passage à l'échelle.

In [126]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import preprocessing
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import learning_curve
from sklearn.feature_selection import SelectKBest, f_classif

## 1. Breast Cancer — classification de tumeurs (bénin / malin)

On commence par le jeu de données le plus simple : des mesures cytologiques de tumeurs du sein, à classer en **bénin** ou **malin**.

### 1.a — Chargement et préparation des données

Avant tout entraînement, on inspecte les données pour répondre à quelques questions qui conditionnent la suite :

- **Valeurs manquantes ?** Le SVM ne les tolère pas, il faudra les traiter.
- **Variables numériques ou catégorielles ?** Le SVM ne travaille que sur du numérique.
- **Variables à la même échelle ?** Sinon il faudra normaliser.
- **Combien de classes dans la cible ?**

L'objectif est d'aboutir à une matrice de features `X` et un vecteur cible `y` entièrement numériques et propres.

In [127]:
# Read in the dataset
cancer = pd.read_csv("breast_cancer.csv")
print("\nLa taille du DataSet est :")
print(cancer.shape)


La taille du DataSet est :
(699, 11)


In [128]:
# Look at the head and dtypes
print("\nLes 5 premières lignes du dataset sont :" )
print(cancer.head())
print("\nLe type des colonnes est :" )
print(cancer.dtypes)

print("\nLes données sont des entiers sauf Bare Nuclei qui est catégorisé en string" )




Les 5 premières lignes du dataset sont :
   Sample_code_number  Clump_Thickness  Uniformity_of_Cell_Size  \
0             1000025                5                        1   
1             1002945                5                        4   
2             1015425                3                        1   
3             1016277                6                        8   
4             1017023                4                        1   

   Uniformity_of_Cell_Shape  Marginal_Adhesion  Single_Epithelial_Cell_Size  \
0                         1                  1                            2   
1                         4                  5                            7   
2                         1                  1                            2   
3                         8                  1                            3   
4                         1                  3                            2   

  Bare_Nuclei  Bland_Chromatin  Normal_Nucleoli  Mitoses  Class  
0           1 

In [129]:
# Convert Bare_Nuclei to numeric and change missing values to NaN
cancer["Bare_Nuclei"] = pd.to_numeric(cancer["Bare_Nuclei"], errors="coerce")
print(cancer.dtypes)

Sample_code_number               int64
Clump_Thickness                  int64
Uniformity_of_Cell_Size          int64
Uniformity_of_Cell_Shape         int64
Marginal_Adhesion                int64
Single_Epithelial_Cell_Size      int64
Bare_Nuclei                    float64
Bland_Chromatin                  int64
Normal_Nucleoli                  int64
Mitoses                          int64
Class                            int64
dtype: object


In [130]:
# Drop the rows that contain a missing value in Bare_Nuclei

print("\nLa taille du dataset avant de retirer les valeurs nulles de Bare_Nuclei")
print(cancer.shape)
print("\nIl y a 16 valeurs nulles :") 
print(cancer["Bare_Nuclei"].isnull().sum())
cancer = cancer.dropna(subset=["Bare_Nuclei"])
print("\nLa taille du dataset après avoir retirer les valeurs nulles de Bare_Nuclei")
print(cancer.shape)



La taille du dataset avant de retirer les valeurs nulles de Bare_Nuclei
(699, 11)

Il y a 16 valeurs nulles :
16

La taille du dataset après avoir retirer les valeurs nulles de Bare_Nuclei
(683, 11)


In [131]:
# Map the Class to be 0 for benign and 1 for Malignant
print("\nLes premieres valeurs de 'Class' sont:") 
print(cancer["Class"].head(10))
print("\nLes valeurs de 'Class' sont:")
print(cancer['Class'].value_counts())
print("\nClass contient soit 2 soit 4") 
cancer["Class"] = cancer["Class"].map({ 2 : 0, 4 : 1 }) #2=benign=0, 4=malignant=1
print("'Class' comporte bien que des 0 et des 1 désormais:") 
print(cancer["Class"])
print("\nLe nombre de valeurs nulles de 'Class' est : ")
print(cancer["Class"].isna().sum())


Les premieres valeurs de 'Class' sont:
0    2
1    2
2    2
3    2
4    2
5    4
6    2
7    2
8    2
9    2
Name: Class, dtype: int64

Les valeurs de 'Class' sont:
Class
2    444
4    239
Name: count, dtype: int64

Class contient soit 2 soit 4
'Class' comporte bien que des 0 et des 1 désormais:
0      0
1      0
2      0
3      0
4      0
      ..
694    0
695    0
696    1
697    1
698    1
Name: Class, Length: 683, dtype: int64

Le nombre de valeurs nulles de 'Class' est : 
0


### 1.b — Construction et comparaison des modèles

On entraîne les modèles et on compare leurs performances :

- On établit d'abord une **baseline** (score obtenu en prédisant toujours la classe majoritaire) comme point de référence.
- On entraîne un **SVM linéaire**, évalué en **validation croisée 3-fold** pour un score robuste qui ne dépend pas d'un seul découpage.
- On répète avec un **noyau RBF**, capable de capturer des frontières non linéaires, et on compare.
- On **normalise** ensuite les données pour vérifier si cela améliore le score — le SVM étant basé sur des distances, l'échelle des variables compte.
- On retient le **meilleur modèle**, puis on l'évalue sur un jeu de test séparé via une **matrice de confusion** et un **rapport de classification** (précision, rappel, f1) pour voir le détail des erreurs.

On utilise un découpage **stratifié** (`stratify=y`) pour conserver la proportion bénin/malin dans le train et le test.

In [132]:
# Set our feature and target variables

X = cancer.drop(columns=["Class", "Sample_code_number"])
y = cancer["Class"]
print("\nLa taille de X est:")
print(X.shape)
print("\nLa taille de y est:")
print(y.shape)



La taille de X est:
(683, 9)

La taille de y est:
(683,)


In [133]:
# Initialize a linear SVC and fit the data
SVC_linear = SVC(kernel='linear', random_state=42)


# Check the model's score
cv_scores = cross_val_score(SVC_linear, X, y, cv=3)
print("\nLinear SVC : Cross Validation Scores:")
print(cv_scores)
print("\nLinear SVC : Mean CV Score:")
print(cv_scores.mean())
print("\nLinear SVC : Std CV Score:")
print(cv_scores.std())

print("\nLe score moyen de validation croisée est élevé et l’écart-type faible, ce qui indique de bonnes performances et une stabilité du modèle\n")



Linear SVC : Cross Validation Scores:
[0.94298246 0.96491228 0.98678414]

Linear SVC : Mean CV Score:
0.9648929592704228

Linear SVC : Std CV Score:
0.017881968169992327

Le score moyen de validation croisée est élevé et l’écart-type faible, ce qui indique de bonnes performances et une stabilité du modèle



In [134]:
# Initialize a RBF SVC and fit the data
SVC_rbf = SVC(kernel='rbf',random_state=42)

# Check the model's score:
cv_scores = cross_val_score(SVC_rbf, X, y, cv=3)
print("\nRbf SVC : Cross Validation Scores:")
print(cv_scores)
print("\nRbf SVC : Mean CV Score:")
print(cv_scores.mean())
print("\nRbf SVC : Std CV Score:")
print(cv_scores.std())

print("\nLes scores de validation croisée sont également élevés, avec une moyenne d'environ 0.96, ce qui indique de très bonnes performances de classification."
"L'écart-type est légèrement plus élevé que pour le SVM linéaire, traduisant une variabilité un peu plus importante entre les différents folds, mais restant globalement faible.\n")


Rbf SVC : Cross Validation Scores:
[0.93421053 0.96491228 0.99118943]

Rbf SVC : Mean CV Score:
0.9634374114434398

Rbf SVC : Std CV Score:
0.023284905256452028

Les scores de validation croisée sont également élevés, avec une moyenne d'environ 0.96, ce qui indique de très bonnes performances de classification.L'écart-type est légèrement plus élevé que pour le SVM linéaire, traduisant une variabilité un peu plus importante entre les différents folds, mais restant globalement faible.



In [135]:
# Now we normalize our data and repeat our models
# Checked both l1 and l2 and saw that l2 performed better, so use l2 for our models below

X_normalized = preprocessing.normalize(X, norm='l2')



In [136]:
# Initialize a linear SVC and fit the data
SVC_linear_normalized = SVC(kernel='linear', random_state=42)

# Check the model's score
cv_scores = cross_val_score(SVC_linear_normalized, X_normalized, y, cv=3)
print("\nLinear SVC normalized: Cross Validation Scores:")
print(cv_scores)
print("\nLinear SVC normalized : Mean CV Score:")
print(cv_scores.mean())
print("/\nLinear SVC normalized : Std CV Score:")
print(cv_scores.std())

print("\nLa version normalisée montre une baisse de précision, ce qui suggère que la normalisation n'est pas bénéfique pour ce jeu de données ")



Linear SVC normalized: Cross Validation Scores:
[0.84649123 0.87280702 0.9030837 ]

Linear SVC normalized : Mean CV Score:
0.8741273153515213
/
Linear SVC normalized : Std CV Score:
0.023122634970564895

La version normalisée montre une baisse de précision, ce qui suggère que la normalisation n'est pas bénéfique pour ce jeu de données 


In [137]:
# Initialize a RBF SVC and fit the data
SVC_rbf_normalized = SVC(kernel='rbf',random_state=42)

# Check the model's score:
cv_scores = cross_val_score(SVC_rbf_normalized, X_normalized, y, cv=3)
print("\nRbf SVC normalized: Cross Validation Scores:")
print(cv_scores)
print("\nRbf SVC normalized: Mean CV Score:")
print(cv_scores.mean())
print("\nRbf SVC normalized: Std CV Score:")
print(cv_scores.std())

print("\nLe SVC RBF normalisé reste performant, mais moins que sa version non normalisée, ce qui montre que la normalisation n'est pas bénéfique pour ce modèle sur ce dataset")


Rbf SVC normalized: Cross Validation Scores:
[0.87719298 0.89035088 0.93832599]

Rbf SVC normalized: Mean CV Score:
0.9019566169461833

Rbf SVC normalized: Std CV Score:
0.026272052141776137

Le SVC RBF normalisé reste performant, mais moins que sa version non normalisée, ce qui montre que la normalisation n'est pas bénéfique pour ce modèle sur ce dataset


In [138]:
# Split the data for final evaluation
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, stratify=y, test_size=0.33, random_state=42)

In [139]:
# Fit the best model (linear non normalisé)
best_model = SVC(kernel='linear', random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

In [140]:
# Print a confusion matrix and classification report for our best performing model:

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[131  16]
 [ 14  65]]
              precision    recall  f1-score   support

           0       0.90      0.89      0.90       147
           1       0.80      0.82      0.81        79

    accuracy                           0.87       226
   macro avg       0.85      0.86      0.85       226
weighted avg       0.87      0.87      0.87       226



**Faux positifs vs faux négatifs.** Dans un contexte médical, les erreurs n'ont pas la même gravité : un **faux négatif** (tumeur maligne classée bénigne) est bien plus dangereux qu'un faux positif. On regarde donc lesquels dominent.

In [141]:
print("Le modèle génère légèrement plus de faux positifs que de faux négatifs, mais ce sont les faux négatifs qui restent les plus critiques dans un contexte médical, car ils correspondent à des cas malins non détectés.")

Le modèle génère légèrement plus de faux positifs que de faux négatifs, mais ce sont les faux négatifs qui restent les plus critiques dans un contexte médical, car ils correspondent à des cas malins non détectés.


### 1.c — Sélection de variables

Toutes les variables n'apportent pas la même information. On utilise `SelectKBest` avec le test **ANOVA F (`f_classif`)** pour ne garder que les variables les plus discriminantes par rapport à la cible.

L'intérêt : un modèle plus simple, plus rapide et souvent plus robuste. On refait la validation croisée avec seulement les 5 meilleures variables pour vérifier que le score se maintient — si oui, on peut se passer du reste sans perte.

In [142]:
# Define a function to select the k best features
def select_k_best_features(X, y, k): 
    selector = SelectKBest(score_func=f_classif, k=k) 
    X_new = selector.fit_transform(X, y) 
    selected_features = X.columns[selector.get_support()] 
    return X_new, selected_features

In [143]:
# Select the 5 best features from X
X_kbest, selected_features = select_k_best_features(X, y, k=5) 
print("Les 5 meilleures features sélectionnées :") 
print(selected_features)

Les 5 meilleures features sélectionnées :
Index(['Uniformity_of_Cell_Size', 'Uniformity_of_Cell_Shape', 'Bare_Nuclei',
       'Bland_Chromatin', 'Normal_Nucleoli'],
      dtype='object')


In [144]:
# Repeat the cross validation with only those 5 features:
# Linear:
SVC_linear_k = SVC(kernel='linear', random_state=42) 
cv_scores_linear_k = cross_val_score(SVC_linear_k, X_kbest, y, cv=3) 
print("\nLinear SVC (5 features) : Cross Validation Scores:")
print(cv_scores_linear_k) 
print("\nMean CV Score:") 
print(cv_scores_linear_k.mean()) 
print("\nStd CV Score:") 
print(cv_scores_linear_k.std())

# RBF SVC
SVC_rbf_k = SVC(kernel='rbf', random_state=42)
cv_scores_rbf_k = cross_val_score(SVC_rbf_k, X_kbest, y, cv=3)

print("\nRBF SVC (5 features) : Cross Validation Scores:")
print(cv_scores_rbf_k)
print("\nMean CV Score:")
print(cv_scores_rbf_k.mean())
print("\nStd CV Score:")
print(cv_scores_rbf_k.std())

print("Les deux modèles conservent d'excellentes performances avec seulement cinq variables, avec une moyenne d'environ 0.96, montrant que la réduction de features n'a quasiment pas dégradé la qualité du SVM")


Linear SVC (5 features) : Cross Validation Scores:
[0.93421053 0.96052632 0.98678414]

Mean CV Score:
0.9605069943581421

Std CV Score:
0.021463092654114165

RBF SVC (5 features) : Cross Validation Scores:
[0.93421053 0.96491228 0.98678414]

Mean CV Score:
0.9619689826622356

Std CV Score:
0.02156375805129877
Les deux modèles conservent d'excellentes performances avec seulement cinq variables, avec une moyenne d'environ 0.96, montrant que la réduction de features n'a quasiment pas dégradé la qualité du SVM


### 1.d — Optimisation des hyperparamètres (Grid Search)

Plutôt que de tester les combinaisons à la main, on automatise la recherche avec `GridSearchCV`. On explore différents **noyaux** et différentes valeurs de **C** (le paramètre de régularisation : un C faible tolère plus d'erreurs mais généralise mieux, un C élevé colle davantage aux données d'entraînement).

Objectif : trouver la combinaison qui maximise le score en validation croisée, et voir si on dépasse les résultats précédents.

In [145]:
param_grid = { 'kernel': ['linear', 'rbf', 'poly'], 'C': [0.1, 1, 10, 100] }
# Modèle SVM 
svc = SVC(random_state=42)
grid_search = GridSearchCV(svc, param_grid, cv=3)
grid_search.fit(X, y) 
print("Meilleurs paramètres trouvés :") 
print(grid_search.best_params_)
print("\nMeilleur score obtenu :") 
print(grid_search.best_score_)

print("\nLe Grid Search a identifié le modèle SVM RBF avec C=0.1 comme la meilleure configuration, atteignant un score de 0.9692, "
"supérieur à nos précédents résultats.")

Meilleurs paramètres trouvés :
{'C': 0.1, 'kernel': 'rbf'}

Meilleur score obtenu :
0.969285364659814

Le Grid Search a identifié le modèle SVM RBF avec C=0.1 comme la meilleure configuration, atteignant un score de 0.9692, supérieur à nos précédents résultats.


## 2. Factorisation : des fonctions réutilisables

Les jeux de données suivants sont plus difficiles. Pour éviter de réécrire le même code à chaque fois, on encapsule les trois étapes récurrentes — validation croisée, évaluation, grid search — dans des fonctions réutilisables. L'analyse devient plus lisible et plus rapide à répéter.

### 2.a — Fonction de validation croisée

`do_cv(model, X, y, cv)` calcule les scores de validation croisée, les affiche, et renvoie la **moyenne** et l'**écart-type** — ce dernier indiquant la stabilité du modèle selon les découpages.

In [ ]:
def do_cv(model, X, y, cv):
    scores = cross_val_score(model, X, y, cv=cv)

    # Affichage du modèle et des résultats
    print("\nModel :", model)
    print("Cross Validation Scores :", scores)
    print("Mean CV Score :", scores.mean())
    print("Std CV Score :", scores.std())

    # Renvoie la moyenne et l'écart-type
    return scores.mean(), scores.std()

### 2.b — Matrice de confusion et rapport de classification

`do_cm_cr(model, X, y, names)` automatise l'évaluation finale : découpage stratifié, entraînement, puis affichage lisible de la matrice de confusion et du rapport de classification. `names` contient les noms des classes pour rendre la sortie compréhensible.

In [147]:
def do_cm_cr(model, X, y, names):
     X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.33, random_state=42 )

     # Fit du modèle
     model.fit(X_train, y_train)

     # Prédictions
     y_pred = model.predict(X_test) 

     print("\nModel :", model)
     print("\nConfusion Matrix :") 
     print(confusion_matrix(y_test, y_pred))

     print("\nClassification Report :") 
     print(classification_report(y_test, y_pred, target_names=names))


### 2.c — Recherche d'hyperparamètres

`do_grid_search(model, parameters)` automatise le grid search : elle teste les combinaisons fournies, affiche les meilleurs paramètres et le meilleur score, et renvoie le meilleur estimateur prêt à l'emploi.

In [148]:
def do_grid_search(model, parameters):

    grid = GridSearchCV(model, parameters, cv=3) 
    grid.fit(X, y)

    print("Meilleurs paramètres trouvés :") 
    print(grid.best_params_) 
    print("\nMeilleur score obtenu :")
    print(grid.best_score_) 
    
    # Return best estimator 
    return grid.best_estimator_


## 3. Spambase — détection de spam

Ce jeu de données contient beaucoup plus de variables (fréquences de mots et de caractères dans des e-mails), à classer en **spam / non spam**. Avec autant de features, l'enjeu principal devient la **sélection de variables** : on identifie les 15 plus pertinentes, puis on lance un grid search pour trouver le meilleur modèle. On réutilise ici les fonctions définies plus haut.

In [149]:
# Load the dataset
spam = pd.read_csv("spambase.csv")
print(spam.head())

   word_freq_make  word_freq_address  word_freq_all  word_freq_3d  \
0            0.00               0.64           0.64           0.0   
1            0.21               0.28           0.50           0.0   
2            0.06               0.00           0.71           0.0   
3            0.00               0.00           0.00           0.0   
4            0.00               0.00           0.00           0.0   

   word_freq_our  word_freq_over  word_freq_remove  word_freq_internet  \
0           0.32            0.00              0.00                0.00   
1           0.14            0.28              0.21                0.07   
2           1.23            0.19              0.19                0.12   
3           0.63            0.00              0.31                0.63   
4           0.63            0.00              0.31                0.63   

   word_freq_order  word_freq_mail  ...  char_freq_;  char_freq_(  \
0             0.00            0.00  ...         0.00        0.000   
1 

In [150]:
# Set the target and feature
X = spam.drop(columns=["class"])
y = spam["class"]


In [151]:
# Select the 15 best features
selector = SelectKBest(score_func=f_classif, k=15) 
X_15 = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()] 

print("Les 15 meilleures features sélectionnées :") 
print(selected_features)

Les 15 meilleures features sélectionnées :
Index(['word_freq_our', 'word_freq_over', 'word_freq_remove',
       'word_freq_order', 'word_freq_receive', 'word_freq_free',
       'word_freq_business', 'word_freq_you', 'word_freq_your',
       'word_freq_000', 'word_freq_hp', 'word_freq_hpl', 'char_freq_!',
       'char_freq_$', 'capital_run_length_total'],
      dtype='object')


In [152]:
# Perform grid search to determine best model

param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.01, 0.1, 1, 10, 100]
}

best_model = do_grid_search(SVC(random_state=42), param_grid)


KeyboardInterrupt: 

In [ ]:
# Print confusion matrix and classification report
do_cm_cr(best_model, X, y, names=['Not Spam', 'Spam'])

In [ ]:
# Perform a CV on this model
do_cv(best_model, X_15, y, cv=3)

## 4. Car Evaluation — variables catégorielles et cible multi-classes

Changement de contexte : ici les variables sont **catégorielles** (prix, nombre de portes, capacité…) et non numériques. Deux adaptations sont nécessaires :

- **Encoder les variables** pour les rendre numériques (le SVM ne traite que du numérique).
- Gérer une **cible à 4 classes** (acceptabilité du véhicule) et non plus binaire, ce qui change la stratégie de modélisation et la lecture des métriques.

In [ ]:
# Read in the dataset car.csv
car = pd.read_csv("car.csv")
car.head()


,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [ ]:
# Analyse exploratoire : toutes les variables sont catégorielles
print("Valeurs uniques par colonne :\n")
for col in car.columns:
    print(f"{col:12s} : {car[col].unique()}")

print("\nValeurs manquantes :", car.isnull().sum().sum())

In [ ]:
# Set the target and features

print("Shape of dataset:", car.shape)
print("\nColumns:")
print(car.columns)

print("\nDataset info:")
print(car.info())


In [ ]:
# Les variables sont catégorielles : on les transforme en variables indicatrices (one-hot).
# Avec get_dummies, les modalités comme 'doors=5more' ou 'persons=more' deviennent
# simplement des colonnes binaires, donc pas besoin de les recoder manuellement.
# La cible (dernière colonne) est encodée en entiers via LabelEncoder.

target_col = car.columns[-1]
print("Colonne cible :", target_col)
print("Classes de la cible :", car[target_col].unique())

# Features : one-hot encoding de toutes les colonnes explicatives
X = pd.get_dummies(car.drop(columns=[target_col]))

# Cible : encodage entier des 4 classes (unacc, acc, good, vgood)
le_car = preprocessing.LabelEncoder()
y = le_car.fit_transform(car[target_col])

print("\nDimensions après encodage :")
print("X :", X.shape, "| nombre de features :", X.shape[1])
print("Correspondance des classes :", dict(zip(le_car.classes_, le_car.transform(le_car.classes_))))

In [ ]:
# Grid search pour déterminer le meilleur modèle (noyau + C)
param_grid = {
    'kernel': ['linear', 'rbf', 'poly'],
    'C': [0.1, 1, 10, 100]
}
best_model_car = do_grid_search(SVC(random_state=42), param_grid)

In [ ]:
# Sélection des 10 meilleures features (test ANOVA F) puis nouveau grid search
selector10 = SelectKBest(score_func=f_classif, k=10)
X_10 = selector10.fit_transform(X, y)
print("10 meilleures features :")
print(list(X.columns[selector10.get_support()]))

grid_car_10 = GridSearchCV(SVC(random_state=42), param_grid, cv=3)
grid_car_10.fit(X_10, y)
print("\nMeilleurs paramètres :", grid_car_10.best_params_)
print("Meilleur score        :", grid_car_10.best_score_)

In [ ]:
# Sélection des 5 meilleures features puis grid search
selector5 = SelectKBest(score_func=f_classif, k=5)
X_5 = selector5.fit_transform(X, y)
print("5 meilleures features :")
print(list(X.columns[selector5.get_support()]))

grid_car_5 = GridSearchCV(SVC(random_state=42), param_grid, cv=3)
grid_car_5.fit(X_5, y)
print("\nMeilleurs paramètres :", grid_car_5.best_params_)
print("Meilleur score        :", grid_car_5.best_score_)

In [ ]:
# Évaluation du meilleur modèle (features complètes) : matrice de confusion + rapport
do_cm_cr(best_model_car, X, y, names=list(le_car.classes_))

In [ ]:
# Validation croisée du meilleur modèle
do_cv(best_model_car, X, y, cv=3)

## 5. Mushroom — encodage catégoriel à grande échelle

Dernier jeu de données : classer des champignons en **comestible / toxique**. Toutes les variables sont catégorielles et le volume est important. On encode chaque variable en valeurs numériques, puis on teste si la **sélection de variables** permet de simplifier le modèle sans dégrader les performances sur un dataset de cette taille.

In [ ]:
# Read in the mushroom data
mush = pd.read_csv("mushroom.csv")
mush.head()


,class,cap-shape,cap-surface,cap-color,bruises?,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [ ]:
# Map the class column (e = edible, p = poisonous)

y = mush["class"].map({"e": 0, "p": 1})

print(y.value_counts())


class
0    4208
1    3916
Name: count, dtype: int64


In [ ]:
# Look at the data

print("Shape of dataset:", mush.shape)

print("\nDataset info:")
print(mush.info())


Shape of dataset: (8124, 23)

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   class                     8124 non-null   object
 1   cap-shape                 8124 non-null   object
 2   cap-surface               8124 non-null   object
 3   cap-color                 8124 non-null   object
 4   bruises?                  8124 non-null   object
 5   odor                      8124 non-null   object
 6   gill-attachment           8124 non-null   object
 7   gill-spacing              8124 non-null   object
 8   gill-size                 8124 non-null   object
 9   gill-color                8124 non-null   object
 10  stalk-shape               8124 non-null   object
 11  stalk-root                8124 non-null   object
 12  stalk-surface-above-ring  8124 non-null   object
 13  stalk-surface-below-ring  8124 non

In [ ]:
# Encodage : chaque modalité (une lettre) est convertie en entier.
# '?' (valeur manquante de la colonne stalk-root) est encodé en -1.
letter_map = {
    'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6,
    'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13,
    'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20,
    'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '?': -1
}

mush_numeric = mush.apply(lambda col: col.map(lambda v: letter_map.get(v, -1)))
print("Aperçu des données encodées :")
print(mush_numeric.head())

In [ ]:
# Cible : comestible (e) = 0, toxique (p) = 1
y = mush["class"].map({"e": 0, "p": 1})

# Features : toutes les colonnes encodées sauf la cible
X = mush_numeric.drop(columns=["class"])

print("X :", X.shape, "| y :", y.shape)
print("Répartition des classes :\n", y.value_counts())

In [ ]:
# Grid search. Le dataset est volumineux (~8000 lignes) : on limite les noyaux
# testés pour garder un temps de calcul raisonnable.
param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10]
}
best_model_mush = do_grid_search(SVC(random_state=42), param_grid)

In [ ]:
# do_grid_search renvoie déjà le meilleur estimateur, réentraîné sur tout le jeu
final_model_mush = best_model_mush
print(final_model_mush)

In [ ]:
# Validation croisée du meilleur modèle
do_cv(final_model_mush, X, y, cv=3)

In [ ]:
# Matrice de confusion + rapport de classification
do_cm_cr(final_model_mush, X, y, names=['Comestible', 'Toxique'])

In [ ]:
# On regarde si 10 features suffisent à conserver de bonnes performances
selector_mush = SelectKBest(score_func=f_classif, k=10)
X_10 = selector_mush.fit_transform(X, y)
print("10 meilleures features :")
print(list(X.columns[selector_mush.get_support()]))

In [ ]:
# Grid search sur les 10 meilleures features
grid_mush_10 = GridSearchCV(SVC(random_state=42), param_grid, cv=3)
grid_mush_10.fit(X_10, y)
print("Meilleurs paramètres :", grid_mush_10.best_params_)
print("Meilleur score        :", grid_mush_10.best_score_)

In [ ]:
# Meilleur modèle sur les 10 features
final_model_mush_10 = grid_mush_10.best_estimator_
print(final_model_mush_10)

In [ ]:
# Validation croisée sur les 10 features
do_cv(final_model_mush_10, X_10, y, cv=3)

In [ ]:
# Matrice de confusion + rapport sur le modèle à 10 features
do_cm_cr(final_model_mush_10, X_10, y, names=['Comestible', 'Toxique'])